# AF2R0 — matched zero-information control
Jalankan setelah static audit PASS. Notebook dapat di-resume dan tidak menampilkan progress bar training ke browser.


In [ ]:
from google.colab import drive
drive.mount('/content/drive',force_remount=True)
import importlib,json,os,shutil,subprocess,sys,tarfile,time,torch
from pathlib import Path
assert torch.cuda.is_available(),'Aktifkan T4 GPU.'
ARM='AF2R0'; REPO=Path('/content/coffee-bean-detection'); BRANCH='agent/af2-adaptive-residual-gate'
if REPO.exists(): shutil.rmtree(REPO)
subprocess.run(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO)],check=True)
sys.path.insert(0,str(REPO/'src')); importlib.invalidate_caches(); os.chdir(REPO)
from coffee_detector.drive_project import resolve_drive_project_root,require_project_artifact
REQ=('bundles/faruq-development-v3-grouped.tar','experiments/faruq-v3-breadth-screening-batch-v1/candidates/AFAB/AF2_seed42/weights/best.pt','experiments/faruq-v3-af2-adaptive-residual-v1/static_audit.json')
PROJECT=resolve_drive_project_root(required_relative_paths=REQ); ARCHIVE=require_project_artifact(PROJECT,REQ[0]); AF2=require_project_artifact(PROJECT,REQ[1]); STATIC=require_project_artifact(PROJECT,REQ[2])
DATA=Path('/content/faruq-development-v3-grouped')
if not (DATA/'data.yaml').is_file():
    with tarfile.open(ARCHIVE,'r') as archive: archive.extractall('/content',filter='data')
assert not (DATA/'test').exists(); OUTPUT=PROJECT/'experiments/faruq-v3-af2-adaptive-residual-v1'
print('GPU:',torch.cuda.get_device_name(0)); print('PROJECT:',PROJECT); print('ARM:',ARM)


In [ ]:
command=[sys.executable,'-u','-m','coffee_detector.experiments.run_faruq_v3_af2r_arm','--arm',ARM,'--data-root',str(DATA),'--grouped-summary',str(DATA/'faruq_grouped_summary.json'),'--af2-checkpoint',str(AF2),'--static-audit',str(STATIC),'--output-root',str(OUTPUT),'--seed','42','--device','0','--authorize-training']
log=OUTPUT/f'{ARM}_seed42_run.log'; log.parent.mkdir(parents=True,exist_ok=True)
print('MENJALANKAN:', ' '.join(command),flush=True); handle=log.open('a',encoding='utf-8'); process=subprocess.Popen(command,cwd=REPO,stdout=handle,stderr=subprocess.STDOUT)
while process.poll() is None:
    csv=OUTPUT/ARM/f'{ARM}_seed42/results.csv'; epochs=max(0,len(csv.read_text(errors='replace').splitlines())-1) if csv.is_file() else 0
    print(f'{ARM}: epoch tercatat {epochs}/30 | log={log}',flush=True); time.sleep(300)
handle.close(); code=process.wait()
if code: print('\n'.join(log.read_text(errors='replace').splitlines()[-120:])); raise RuntimeError(f'{ARM} gagal: {code}')
result=json.loads((OUTPUT/'val_reports'/f'{ARM}_seed42_result.json').read_text()); print(json.dumps(result,indent=2))
